AI VISION LAB · 05 / STUDENT WORKBOOK

# Mission 07. 회전 각도별 CNN 성능 비교하기

이미지 데이터 증강과 재현 가능한 실험

**학습 목표**

- 저장한 CNN 모델을 다시 불러와 동일한 테스트 데이터로 평가할 수 있다.
- 정확도와 클래스별 Precision·Recall을 비교하고 혼동행렬을 해석할 수 있다.
- 전체 정확도와 특정 클래스의 오분류 변화가 서로 다를 수 있음을 설명할 수 있다.

### VS Code 실행 준비

`DATA_ROOT`에 4강 `Test`가 필요합니다. 미션 06에서 `models`에 저장한 네 모델을 같은 테스트 데이터로 평가합니다.

커널 목록에서 TensorFlow가 설치된 Python 환경을 선택하고 위에서부터 실행하세요.

In [ ]:
from pathlib import Path
from tensorflow import keras
from IPython.display import display

COURSE_DIR = Path('D:\\2604340036 - 송성민\\2학기\\반도체장비AI비전실무\\2026-AIVISION_05강')
DATA_ROOT = COURSE_DIR.parent / "wafermap_image_data_261021"

IMG_SIZE = 64
class_names = ["Center", "Edge-Loc", "Edge-Ring", "Loc", "Scratch"]
BATCH_SIZE = 32
TEST_DIR = DATA_ROOT / "Test"
MODEL_DIR = COURSE_DIR / "models"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

def check_image_folders(directory):
    missing = [
        name for name in class_names
        if not any(
            path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            for path in (directory / name).rglob("*")
        )
    ]
    if missing:
        raise FileNotFoundError(
            f"{directory}에 이미지가 없는 클래스: {', '.join(missing)}. "
            "첫 셀의 DATA_ROOT를 4강 Train/Test가 있는 폴더로 설정하세요."
        )

In [ ]:
import numpy as np
import tensorflow as tf

def list_images(directory):
    check_image_folders(directory)
    image_paths = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_paths = sorted(
            path for path in (directory / class_name).rglob("*")
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        image_paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return image_paths, np.asarray(labels, dtype=np.int32)

def decode_image(image_bytes, label):
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

def make_dataset(image_paths, labels, shuffle=False, seed=42):
    # Python이 파일을 읽어 Windows의 한글 경로를 처리합니다.
    image_bytes = [path.read_bytes() for path in image_paths]
    dataset = tf.data.Dataset.from_tensor_slices((image_bytes, labels))
    if shuffle:
        dataset = dataset.shuffle(BATCH_SIZE * 8, seed=seed)
    dataset = dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_paths, test_labels = list_images(TEST_DIR)
test_ds = make_dataset(test_paths, test_labels)
print(f"테스트 이미지: {len(test_paths):,}장")

## 1. 저장한 모델을 다시 평가하는 이유

Mission 06에서는 회전 범위 `0°, ±15°, ±45°, ±90°`로 네 개의 CNN을 학습해 `.keras` 파일로 저장했다. 이번에는 **재학습하지 않고 파일을 불러와 같은 테스트 데이터로 평가**한다.

저장된 CNN 모델 4개cnn_rotation_0.keras · 15 · 45 · 90

↓

같은 test_ds로 예측 및 평가shuffle=False · 같은 클래스 순서

↓

정확도 → 클래스별 지표 → 혼동행렬모델마다 어떤 유형에서 틀리는지 분석

이번 단계에서는 모델을 선택하거나 개선하기보다 **차이를 관찰하고 해석**한다. 실제 모델 개발에서는 검증 데이터로 실험 조건을 선택하고, 테스트 데이터는 최종 평가용으로 남겨두는 것이 원칙이다.

## 2. 모델별 테스트 정확도 비교하기
먼저 네 모델을 동일한 `test_ds`로 평가해 정확도와 손실을 표로 만든다. 아래 코드의 `MODEL_DIR`은 Mission 06에서 사용한 Google Drive 저장 폴더다.

In [ ]:
import pandas as pd

rotation_degrees = [0, 15, 45, 90]
missing_models = [
    MODEL_DIR / f"cnn_rotation_{degree}.keras"
    for degree in rotation_degrees
    if not (MODEL_DIR / f"cnn_rotation_{degree}.keras").is_file()
]
if missing_models:
    names = ", ".join(path.name for path in missing_models)
    raise FileNotFoundError(f"미션 06에서 먼저 저장할 모델: {names}")
results = []

for degree in rotation_degrees:
    filename = MODEL_DIR / f"cnn_rotation_{degree}.keras"
    # 학습 없이 저장된 모델을 다시 불러온다.
    model_exp = keras.models.load_model(filename)

    # 모든 모델을 동일한 테스트 데이터셋으로 평가한다.
    loss, accuracy = model_exp.evaluate(test_ds, verbose=0)
    results.append({
        "회전 각도": degree,
        "정확도": accuracy,
        "손실": loss
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

`load_model()`은 모델을 파일에서 복원하고, `evaluate()`는 지정한 데이터셋의 손실과 정확도를 계산한다. `results` 리스트에 기록한 값을 `DataFrame`으로 변환하면 비교하기 편하다.
**결과 기록**

1. 정확도가 가장 높은 회전 범위를 찾는다.
2. 회전하지 않은 모델과 비교해 정확도가 얼마나 달라졌는지 기록한다.
3. 정확도만 보고 최종적으로 좋은 모델이라고 말할 수 있을지 생각한다.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(results_df["회전 각도"], results_df["정확도"], marker="o")
plt.xticks(rotation_degrees)
plt.ylim(0, 1)
plt.xlabel("Rotation degree (±)")
plt.ylabel("Test accuracy")
plt.grid(True)
plt.show()

## 3. 정확도 외에 무엇을 확인해야 할까?
모델의 정확도가 같아도 **어떤 불량 유형을 틀렸는지**에 따라 의미가 달라진다. 예를 들어 실제 `Loc`을 `Scratch`로 잘못 예측하면 국부 불량을 선형 불량으로 잘못 해석할 수 있다.
| 지표 | 확인하는 질문 |
| --- | --- |
| **Precision (정밀도)** | 모델이 어떤 클래스라고 예측한 것 중 실제로 그 클래스인 비율은? |
| **Recall (재현율)** | 실제 그 클래스인 이미지 중 모델이 찾아낸 비율은? |
| **혼동행렬** | 실제 클래스가 어떤 다른 클래스로 오분류되었는가? |
클래스별 지표를 계산하려면 먼저 **실제 라벨**과 **모델 예측 라벨**을 같은 순서로 준비해야 한다.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# test_ds는 shuffle=False로 생성한 데이터셋
# 라벨을 배치 순서대로 연결
y_true = np.concatenate([
    labels.numpy() for images, labels in test_ds
])

print("테스트 이미지 수:", len(y_true))
print("클래스 순서:", class_names)

**주의:** `test_ds`를 `shuffle=False`로 만들었는지, `class_names`가 테스트 폴더의 클래스 인덱스 순서와 일치하는지 확인한다. 정답과 예측의 순서가 어긋나면 지표가 잘못 계산된다.

## 4. 클래스별 Precision·Recall 비교하기
모델의 `predict()`는 각 클래스의 예측 확률을 출력한다. `np.argmax(..., axis=1)`은 확률이 가장 높은 클래스의 인덱스를 선택한다.

In [ ]:
reports = {}
confusion_matrices = {}

for degree in rotation_degrees:
    filename = MODEL_DIR / f"cnn_rotation_{degree}.keras"
    model_exp = keras.models.load_model(filename)

    # 각 클래스의 예측 확률에서 가장 높은 클래스 인덱스를 선택한다.
    y_prob = model_exp.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    reports[degree] = classification_report(
        y_true, y_pred,
        labels=range(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    confusion_matrices[degree] = confusion_matrix(
        y_true, y_pred,
        labels=range(len(class_names))
    )

    print(f"±{degree}° 분석 완료")

`reports`와 `confusion_matrices`는 회전 각도를 키로 사용하는 딕셔너리다. 예를 들어 `reports[45]`에는 ±45° 모델의 성능지표가 저장된다.
다음 코드는 딕셔너리에서 클래스별 지표를 꺼내 표로 만든다. 기존에 배운 `DataFrame`과 `pivot()`을 활용한다.

In [ ]:
rows = []

for degree in rotation_degrees:
    for class_name in class_names:
        metrics = reports[degree][class_name]
        rows.append({
            "회전 각도": degree,
            "클래스": class_name,
            "Precision": metrics["precision"],
            "Recall": metrics["recall"]
        })

metrics_df = pd.DataFrame(rows)

precision_df = metrics_df.pivot(
    index="클래스", columns="회전 각도", values="Precision"
)
recall_df = metrics_df.pivot(
    index="클래스", columns="회전 각도", values="Recall"
)

print("Precision 비교")
display(precision_df.round(2))
print("Recall 비교")
display(recall_df.round(2))

**결과 관찰**

1. 각도에 따라 재현율 변화가 큰 클래스를 찾는다.
2. 정밀도가 개선되면서 재현율은 나빠진 클래스가 있는지 확인한다.
3. 전체 정확도가 가장 높았던 모델이 모든 클래스에서도 가장 좋은지 확인한다.

## 5. 혼동행렬로 오분류 방향 확인하기
혼동행렬은 **행이 실제 클래스(True), 열이 예측 클래스(Predicted)**이다. 대각선은 정분류이고, 대각선 밖의 숫자는 오분류된 이미지 수다.
**혼동행렬 읽기 예시**

가상 예시: 실제 Loc 20장 중 8장을 Scratch로 예측했다.

**오분류 해석**

실제 Loc

예측 Scratch

Loc → Scratch 오분류

행과 열을 바꾸어 해석하면 다른 오류가 된다. 어떤 클래스를 놓쳤는지 구분하자.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, degree in zip(axes, rotation_degrees):
    sns.heatmap(
        confusion_matrices[degree],
        annot=True, fmt="d", cmap="viridis",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=False, ax=ax
    )
    ax.set_title(f"Rotation ±{degree}°")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.show()

출력된 그림에서 대각선 밖 숫자가 큰 칸을 찾아보자. 동일한 클래스 쌍이라도 어느 방향으로 잘못 예측했는지에 따라 의미가 달라진다.

## 6. Loc → Scratch 오분류 집중 분석
전체 정확도가 낮아졌다는 사실만으로는 무슨 문제가 생겼는지 알 수 없다. 이번에는 **실제 Loc을 Scratch로 잘못 분류한 개수**가 회전 범위에 따라 어떻게 변하는지 확인해보자.

In [ ]:
loc_idx = class_names.index("Loc")
scratch_idx = class_names.index("Scratch")

loc_results = []
for degree in rotation_degrees:
    cm = confusion_matrices[degree]
    loc_results.append({
        "회전 각도": degree,
        "Loc 정분류": cm[loc_idx, loc_idx],
        "Loc → Scratch": cm[loc_idx, scratch_idx]
    })

loc_df = pd.DataFrame(loc_results)
display(loc_df)

`cm[loc_idx, scratch_idx]`는 실제 클래스가 `Loc`인 행과 예측 클래스가 `Scratch`인 열이 만나는 값이다.

회전 범위에 따라 오분류 개수가 변하더라도 그것만으로 **회전 증강이 원인이라고 확정할 수는 없다.** 데이터 수, 모델 학습의 변동성, 다른 클래스의 성능 변화도 함께 고려해야 한다.

## 7. 코드 이해 및 실험 결과 분석

**Q7.1.**

`load_model()`을 사용하면 어떤 작업을 다시 하지 않아도 되는가? 모델을 저장해두는 이유와 연결해 설명하시오.

**답변**

`load_model()`로 저장한 구조와 가중치를 복원하면 **학습을 다시 하지 않고** 예측과 평가를 할 수 있습니다.

**Q7.2.**

`test_ds`를 `shuffle=False`로 만들어야 하는 이유를 `y_true`와 `y_pred`의 순서에 연결하여 설명하시오.

**답변**

`y_true`와 `y_pred`는 같은 이미지 순서여야 합니다. `shuffle=False`로 두 번 순회할 때 순서를 유지해야 각 예측과 해당 정답을 올바르게 비교할 수 있습니다.

**Q7.3.**

정확도가 가장 높은 회전 범위는 무엇인가? 회전 증강을 하지 않은 모델과 정확도를 비교하여 기록하시오.

**답변**

**검증 예시(테스트 15장, 모델별 1에포크)**에서 0°·±15°·±45°·±90° 모두 **20.00%**로 공동 최고였고, 0°와의 차이는 **0%p**였습니다. 수업 데이터의 결과는 `results_df`에서 확인하세요.

**Q7.4.**

클래스 한 개를 선택하여 회전 각도별 Precision과 Recall을 비교하시오. 두 지표가 같은 방향으로 변했는가?

**답변**

같은 검증 예시의 **Center**는 0°·±15°·±45°·±90° 순서로 Precision이 **0.20, 0.20, 0.23, 0.00**, Recall이 **1.00, 1.00, 1.00, 0.00**이었습니다. ±45°에서 Precision만 높아졌습니다. 수업 데이터의 값은 `precision_df`와 `recall_df`에서 확인하세요.

**Q7.5.**

혼동행렬에서 가장 눈에 띄는 오분류 방향을 한 가지 찾으시오. 실제 클래스와 예측 클래스를 정확히 구분하여 기록하시오.

**답변**

같은 검증 예시에서 0° 모델은 **실제 Edge-Loc 3장을 Center로 예측**했습니다. 혼동행렬의 Edge-Loc 행과 Center 열에서 확인할 수 있습니다.

**Q7.6.**

`Loc → Scratch` 오분류가 가장 많은 회전 범위는 무엇인가? 이 현상만으로 회전 증강이 Loc 분류를 악화시킨 원인이라고 단정할 수 있는가? 이유를 작성하시오.

**답변**

같은 검증 예시의 `Loc → Scratch`는 **네 각도 모두 0개**로 공동 최대였습니다. 오분류 한 종류만으로 원인을 단정하기는 어려우므로 `loc_df`의 Loc 정분류 수와 다른 오분류 방향, 시드별 결과도 함께 봐야 합니다.